In [1]:
from lutils.stock import LTdxHq
import datetime

In [2]:
from pytdx.exhq import TdxExHq_API
from pytdx.hq import TdxHq_API

In [8]:
from abc import ABCMeta
from multiprocessing import Pool, cpu_count
import time
from concurrent.futures import ThreadPoolExecutor


class Parallelism_abs(object, metaclass=ABCMeta):
    def __init__(self, processes=cpu_count()):
        '''

        :param processes: 进程数量，默认为cpu个数
        '''
        self.total_processes = 0
        self.completed_processes = 0
        self.results = []
        self.data = []
        self.cores = processes  # cpu核心数量
        self._loginfolist = []  # 保存打印信息

    def __getstate__(self):
        self_dict = self.__dict__.copy()
        print(self.__dict__)
        del self_dict['pool']
        return self_dict

    def __setstate__(self, state):
        print(state)
        self.__dict__.update(state)

    def get_results(self):
        return self.results

    def complete(self, result):
        self.results.extend(result)
        self.completed_processes += 1
        print('Progress: {:.2f}%'.format(
            (self.completed_processes / self.total_processes) * 100))


class Parallelism(Parallelism_abs):
    """ 多进程map类
        pl = ParallelSim()
        pl.run(yourFunc, yourIter)
        data = pl.get_results()
        data = list(data)
        print(data)
    """

    def __init__(self, processes=cpu_count()):
        super(Parallelism, self).__init__(processes)
        self.pool = Pool(processes=processes)

    def run(self, func, iter):
        if isinstance(iter, list) and self.cores > 1 and len(
                iter) > self.cores:
            j = self.cores
            print('2-11')
            for i in range(j):
                print('2-22')
                pLen = int(len(iter) / j) + 1
                self.data.append(
                    self.pool.starmap_async(func,
                                            iter[i * pLen:(i + 1) * pLen],
                                            callback=self.complete,
                                            error_callback=self.exception))
                self.total_processes += 1
        else:
            print('2-33')
            self.data.append(
                self.pool.starmap_async(func=func,
                                        iterable=iter,
                                        callback=self.complete,
                                        error_callback=self.exception)
            )
            self.total_processes += 1
        for i in range(self.total_processes):
            print('2-44')
            try:
                while not self.data[i].ready():
                    time.sleep(0.5)
                self.data[i].get()
            except Exception as e:
                print(e.args)
        self.pool.close()
        self.pool.join()

    def exception(self, exception=None):
        print(exception)

In [9]:
def ping(ip, port=7709, type_='stock'):
    api = TdxHq_API()
    apix = TdxExHq_API(raise_exception=True)
    __time1 = datetime.datetime.now()
    print('2222')
    try:
        if type_ in ['stock']:
            with api.connect(ip, port, time_out=0.7):
                res = api.get_security_list(0, 1)

                if res is not None:
                    if len(api.get_security_list(0, 1)) > 800:
                        return datetime.datetime.now() - __time1
                    else:
                        print('BAD RESPONSE {}'.format(ip))
                        return datetime.timedelta(9, 9, 0)
                else:

                    print('BAD RESPONSE {}'.format(ip))
                    return datetime.timedelta(9, 9, 0)
        elif type_ in ['future']:
            with apix.connect(ip, port, time_out=0.7):
                res = apix.get_instrument_count()
                if res is not None:
                    if res > 20000:
                        return datetime.datetime.now() - __time1
                    else:
                        print('️Bad FUTUREIP REPSONSE {}'.format(ip))
                        return datetime.timedelta(9, 9, 0)
                else:
                    print('️Bad FUTUREIP REPSONSE {}'.format(ip))
                    return datetime.timedelta(9, 9, 0)
    except Exception as e:
        if isinstance(e, TypeError):
            print(e)
            print('Tushare内置的pytdx版本和QUANTAXIS使用的pytdx 版本不同, 请重新安装pytdx以解决此问题')
            print('pip uninstall pytdx')
            print('pip install pytdx')

        else:
            print('BAD RESPONSE {}'.format(ip))
        return datetime.timedelta(9, 9, 0)

In [3]:
ltdxhq = LTdxHq()

init tdx ip by ping...
Progress: 6.25%
Progress: 12.50%
Progress: 18.75%
Progress: 25.00%
Progress: 31.25%
Progress: 37.50%
Progress: 43.75%
Progress: 50.00%
Progress: 56.25%
Progress: 62.50%
Progress: 68.75%
Progress: 75.00%
Progress: 81.25%
Progress: 87.50%
Progress: 93.75%
saving ip list to stock cache 33


In [4]:
df = ltdxhq.get_k_data_daily('300417', start='2024-10-08') # 300417 # 603636

In [5]:
df

,open,close,high,low,volume,amount
date,,,,,,
2024-10-08,12.10,11.51,12.10,10.66,11963221.0,134109544.0
2024-10-09,11.00,9.98,11.22,9.98,9014337.0,95477296.0
2024-10-10,10.22,10.35,10.70,9.90,8216670.0,85232424.0
2024-10-11,10.35,9.90,10.37,9.70,6318730.0,62897904.0
2024-10-14,10.80,10.58,10.86,10.03,12829263.0,132954144.0
...,...,...,...,...,...,...
2024-12-25,12.95,12.86,13.14,12.36,5031820.0,64015828.0
2024-12-26,11.86,12.22,12.48,11.52,10175320.0,122751976.0
2024-12-27,12.22,12.79,13.18,12.14,7685870.0,97458432.0


In [5]:
n=0
_type='stock'

In [6]:
ip_list=[("长城国瑞电信1", "218.85.139.19", 7709),
    ("长城国瑞电信2", "218.85.139.20", 7709),
    ("长城国瑞网通", "58.23.131.163", 7709),
    ("上证云成都电信一", "218.6.170.47", 7709),
    ("上证云北京联通一", "123.125.108.14", 7709),
    ("上海电信主站Z1", "180.153.18.170", 7709),
    ("上海电信主站Z2", "180.153.18.171", 7709),
    ("上海电信主站Z80", "180.153.18.172", 80),
    ("北京联通主站Z1", "202.108.253.130", 7709),
    ("北京联通主站Z2", "202.108.253.131", 7709),
    ("北京联通主站Z80", "202.108.253.139", 80),
    ("杭州电信主站J1", "60.191.117.167", 7709),
    ("杭州电信主站J2", "115.238.56.198", 7709),
    ("杭州电信主站J3", "218.75.126.9", 7709),
    ("杭州电信主站J4", "115.238.90.165", 7709),
    ("杭州联通主站J1", "124.160.88.183", 7709),
    ("杭州联通主站J2", "60.12.136.250", 7709),
    ("杭州华数主站J1", "218.108.98.244", 7709),
    ("杭州华数主站J2", "218.108.47.69", 7709),
    ("义乌移动主站J1", "223.94.89.115", 7709),
    ("青岛联通主站W1", "218.57.11.101", 7709),
    ("青岛电信主站W1", "58.58.33.123", 7709),
    ("深圳电信主站Z1", "14.17.75.71", 7709),
    ("云行情上海电信Z1", "114.80.63.12", 7709),
    ("云行情上海电信Z2", "114.80.63.35", 7709),
    ("上海电信主站Z3", "180.153.39.51", 7709),
    ('招商证券深圳行情', '119.147.212.81', 7709),
    ('华泰证券(南京电信)', '221.231.141.60', 7709),
    ('华泰证券(上海电信)', '101.227.73.20', 7709),
    ('华泰证券(上海电信二)', '101.227.77.254', 7709),
    ('华泰证券(深圳电信)', '14.215.128.18', 7709),
    ('华泰证券(武汉电信)', '59.173.18.140', 7709),
    ('华泰证券(天津联通)', '60.28.23.80', 7709),
    ('华泰证券(沈阳联通)', '218.60.29.136', 7709),
    ('华泰证券(南京联通)', '122.192.35.44', 7709),
    ('华泰证券(南京联通)', '122.192.35.44', 7709),
    ('安信', '112.95.140.74', 7709),
    ('安信', '112.95.140.92', 7709),
    ('安信', '112.95.140.93', 7709),
    ('安信', '114.80.149.19', 7709),
    ('安信', '114.80.149.21', 7709),
    ('安信', '114.80.149.22', 7709),
    ('安信', '114.80.149.91', 7709),
    ('安信', '114.80.149.92', 7709),
    ('安信', '121.14.104.60', 7709),
    ('安信', '121.14.104.66', 7709),
    ('安信', '123.126.133.13', 7709),
    ('安信', '123.126.133.14', 7709),
    ('安信', '123.126.133.21', 7709),
    ('安信', '211.139.150.61', 7709),
    ('安信', '59.36.5.11', 7709),
    ('广发', '119.29.19.242', 7709),
    ('广发', '123.138.29.107', 7709),
    ('广发', '123.138.29.108', 7709),
    ('广发', '124.232.142.29', 7709),
    ('广发', '183.57.72.11', 7709),
    ('广发', '183.57.72.12', 7709),
    ('广发', '183.57.72.13', 7709),
    ('广发', '183.57.72.15', 7709),
    ('广发', '183.57.72.21', 7709),
    ('广发', '183.57.72.22', 7709),
    ('广发', '183.57.72.23', 7709),
    ('广发', '183.57.72.24', 7709),
    ('广发', '183.60.224.177', 7709),
    ('广发', '183.60.224.178', 7709),
    ('国泰君安', '113.105.92.100', 7709),
    ('国泰君安', '113.105.92.101', 7709),
    ('国泰君安', '113.105.92.102', 7709),
    ('国泰君安', '113.105.92.103', 7709),
    ('国泰君安', '113.105.92.104', 7709),
    ('国泰君安', '113.105.92.99', 7709),
    ('国泰君安', '117.34.114.13', 7709),
    ('国泰君安', '117.34.114.14', 7709),
    ('国泰君安', '117.34.114.15', 7709),
    ('国泰君安', '117.34.114.16', 7709),
    ('国泰君安', '117.34.114.17', 7709),
    ('国泰君安', '117.34.114.18', 7709),
    ('国泰君安', '117.34.114.20', 7709),
    ('国泰君安', '117.34.114.27', 7709),
    ('国泰君安', '117.34.114.30', 7709),
    ('国泰君安', '117.34.114.31', 7709),
    ('国信', '182.131.3.252', 7709),
    ('国信', '183.60.224.11', 7709),
    ('国信', '58.210.106.91', 7709),
    ('国信', '58.63.254.216', 7709),
    ('国信', '58.63.254.219', 7709),
    ('国信', '58.63.254.247', 7709),
    ('海通', '123.125.108.90', 7709),
    ('海通', '175.6.5.153', 7709),
    ('海通', '182.118.47.151', 7709),
    ('海通', '182.131.3.245', 7709),
    ('海通', '202.100.166.27', 7709),
    ('海通', '222.161.249.156', 7709),
    ('海通', '42.123.69.62', 7709),
    ('海通', '58.63.254.191', 7709),
    ('海通', '58.63.254.217', 7709),
    ('华林', '120.55.172.97', 7709),
    ('华林', '139.217.20.27', 7709),
    ('华林', '202.100.166.21', 7709),
    ('华林', '202.96.138.90', 7709),
    ('华林', '218.106.92.182', 7709),
    ('华林', '218.106.92.183', 7709),
    ('华林', '220.178.55.71', 7709),
    ('华林', '220.178.55.86', 7709),]

In [10]:
# ips = [(x['ip'], x['port'], _type) for x in ip_list]
results = []
ips = [(x[1], x[2], _type) for x in ip_list]
print('11')
ps = Parallelism()
print('22')
ps.run(ping, ips)
print('33')
data = list(ps.get_results())
results = []
for i in range(len(data)):
    # 删除ping不通的数据
    print(data[i])
    if data[i] < datetime.timedelta(0, 9, 0):
        results.append((data[i], ip_list[i]))
# 按照ping值从小大大排序
results = [x[1] for x in sorted(results, key=lambda x: x[0])]
if _type:
    # store the data as binary data stream
    print('saving ip list to {} cache {}'.format(_type, len(results)))

11
22
2-11
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-22
2-44


KeyboardInterrupt: 

In [32]:
ips = list([[x[1], x[2], _type, None] for x in ip_list])

In [33]:
for i, (ip, port, t, _) in enumerate(ips):
    print(ip)
    r = ping(ip, port, t)
    print(r)
    ips[i][3] = r

218.85.139.19
BAD RESPONSE 218.85.139.19
9 days, 0:00:09
218.85.139.20
BAD RESPONSE 218.85.139.20
9 days, 0:00:09
58.23.131.163
BAD RESPONSE 58.23.131.163
9 days, 0:00:09
218.6.170.47
0:00:00.310292
123.125.108.14
0:00:00.362512
180.153.18.170
0:00:00.178080
180.153.18.171
BAD RESPONSE 180.153.18.171
9 days, 0:00:09
180.153.18.172
0:00:00.192026
202.108.253.130
BAD RESPONSE 202.108.253.130
9 days, 0:00:09
202.108.253.131
BAD RESPONSE 202.108.253.131
9 days, 0:00:09
202.108.253.139
0:00:00.384056
60.191.117.167
0:00:00.181227
115.238.56.198
0:00:00.146177
218.75.126.9
0:00:00.179711
115.238.90.165
0:00:00.197804
124.160.88.183
BAD RESPONSE 124.160.88.183
9 days, 0:00:09
60.12.136.250
0:00:00.311229
218.108.98.244
BAD RESPONSE 218.108.98.244
9 days, 0:00:09
218.108.47.69
BAD RESPONSE 218.108.47.69
9 days, 0:00:09
223.94.89.115
BAD RESPONSE 223.94.89.115
9 days, 0:00:09
218.57.11.101
BAD RESPONSE 218.57.11.101
9 days, 0:00:09
58.58.33.123
BAD RESPONSE 58.58.33.123
9 days, 0:00:09
14.17.75

In [34]:
ips

[['218.85.139.19', 7709, 'stock', datetime.timedelta(days=9, seconds=9)],
 ['218.85.139.20', 7709, 'stock', datetime.timedelta(days=9, seconds=9)],
 ['58.23.131.163', 7709, 'stock', datetime.timedelta(days=9, seconds=9)],
 ['218.6.170.47', 7709, 'stock', datetime.timedelta(microseconds=310292)],
 ['123.125.108.14', 7709, 'stock', datetime.timedelta(microseconds=362512)],
 ['180.153.18.170', 7709, 'stock', datetime.timedelta(microseconds=178080)],
 ['180.153.18.171', 7709, 'stock', datetime.timedelta(days=9, seconds=9)],
 ['180.153.18.172', 80, 'stock', datetime.timedelta(microseconds=192026)],
 ['202.108.253.130', 7709, 'stock', datetime.timedelta(days=9, seconds=9)],
 ['202.108.253.131', 7709, 'stock', datetime.timedelta(days=9, seconds=9)],
 ['202.108.253.139', 80, 'stock', datetime.timedelta(microseconds=384056)],
 ['60.191.117.167', 7709, 'stock', datetime.timedelta(microseconds=181227)],
 ['115.238.56.198', 7709, 'stock', datetime.timedelta(microseconds=146177)],
 ['218.75.126.9',

In [ ]:
000034

In [6]:
ltdxhq = LTdxHq()

init tdx ip by ping...
Progress: 6.25%
Progress: 12.50%
Progress: 18.75%
Progress: 25.00%
Progress: 31.25%
Progress: 37.50%
Progress: 43.75%
Progress: 50.00%
Progress: 56.25%
Progress: 62.50%
Progress: 68.75%
Progress: 75.00%
Progress: 81.25%
saving ip list to stock cache 38


In [7]:
df = ltdxhq.get_k_data_daily('000561', start='2024-10-08') # 000031 000035 000035

In [8]:
df

,open,close,high,low,volume,amount
date,,,,,,
2024-10-08,8.72,8.35,8.72,7.82,59937576.0,497112032.0
2024-10-09,7.94,7.51,8.01,7.51,33274900.0,257764224.0
2024-10-10,7.54,7.53,7.79,7.35,20829836.0,158237744.0
2024-10-11,7.52,7.18,7.52,7.08,16391179.0,118934864.0
2024-10-14,7.27,7.58,7.62,7.21,21892982.0,163570288.0
...,...,...,...,...,...,...
2024-12-25,8.33,8.12,8.38,8.02,8736200.0,71063112.0
2024-12-26,8.12,8.25,8.32,8.12,7340901.0,60682536.0
2024-12-27,8.41,8.55,8.88,8.35,21331210.0,183395936.0


In [7]:
import configparser

In [17]:
def update_hosts(hosts_file):
    config = configparser.ConfigParser()
    config.read(hosts_file, encoding='gb2312')

    hq_count = int(config['HQHOST']['HostNum'])

    hq_hosts = []
    for i in range(1, hq_count+1, 1):
        host_key = 'HostName%02d' % i
        ip_key = 'IPAddress%02d' % i
        port_key = 'Port%02d' % i

        h = config['HQHOST'][host_key]
        ip = config['HQHOST'][ip_key]
        port = int(config['HQHOST'][port_key])

        hq_hosts.append((h, ip, port))
    
    return hq_hosts
        

In [18]:
a = update_hosts('D:/tools/new_tdx/connect.cfg')

In [20]:
len(a)

38